In [2]:
import os
print(os.getcwd())

c:\Projects\food_meg_analyses


In [2]:

import sys
import numpy as np
import numbers
import os
import mne
import glob
import warnings
from mne.time_frequency import csd_morlet, read_spectrum, read_csd, read_tfrs
import traceback
from pymatreader import read_mat
from typeguard import typechecked

In [7]:

def compute_psd(evoked_instance: mne.Evoked, fmin: float, fmax: float, tmin: float, tmax: float, picks: str)-> mne.time_frequency.Spectrum:
    # try:
    #     if not isinstance(evoked_instance, mne.evoked.Evoked):
    #         raise TypeError("evoked_instance should be an mne.evoked.Evoked instance, wrong input type was given.")
        
    # except TypeError as e:
    #     print("An error occured:", e)

    # else:
    try:

        psd = evoked_instance.compute_psd(method='morlet', fmin=fmin, fmax=fmax, tmin=tmin, tmax=tmax, picks=picks)

    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()


    if 'psd' not in locals():
        psd = None
    return psd



In [ ]:

freq_bands = [
        (3, 7),     # theta
        (7, 13),    # alpha
        (13, 17),   # low beta
        (17, 25),   # high beta 1
        (25, 30),   # high beta 2
    ]

time_frames = [
    (0, 0.15),
    (0.15, 0.3),
    (0.3, 0.5),
    (0.5, 0.8),
]
# Define event_ids mapping
event_ids = {
    "food/short/rep1": 10, "food/medium/rep1": 12, "food/long/rep1": 14, 
    "food/short/rep2": 20, "food/medium/rep2": 22, "food/long/rep2": 24,
    "positive/short/rep1": 110, "positive/medium/rep1": 112, "positive/long/rep1": 114,
    "positive/short/rep2": 120, "positive/medium/rep2": 122, "positive/long/rep2": 124,
    "neutral/short/rep1": 210, "neutral/medium/rep1": 212, "neutral/long/rep1": 214,
    "neutral/short/rep2": 220, "neutral/medium/rep2": 222, "neutral/long/rep2": 224
}

new_event_ids = {"food_1": 1 , "food_2": 2, "positive_1": 3, "positive_2": 4, "neutral_1": 5, "neutral_2": 6}

# Define combination of conditions
pres_1 = ['food_1', 'positive_1', 'neutral_1']
pres_2 = ['food_2', 'positive_2', 'neutral_2']
food = ['food_1', 'food_2']
positive = ['positive_1', 'positive_2']
neutral = ['neutral_1', 'neutral_2']
nonfood = ['positive_1', 'positive_2', 'neutral_1', 'neutral_2']
nonfood_1 = ['positive_1', 'neutral_1']
nonfood_2 = ['positive_2', 'neutral_2']

#bad + reference channel names
bad_ch_names = ['A17','A203','TRIGGER','RESPONSE','MLzA','MLyA','MLzaA','MLyaA','MLxA','MLxaA','MRzA','MRxA','MRzaA','MRxaA','MRyA',
                'MCzA','MRyaA','MCzaA','MCyA','GzxA','MCyaA','MCxA','MCxaA','GyyA','GzyA','GxxA','GyxA','UACurrent','X1','X3','X5','X2','X4','X6']

channels_number = 246

trial_number = 595

time_points = 1119

oddball_id = 8

baseline_time = (-0.3, 0)

post_stim_time = (0, 0.8)

project_directory = "C:/Projects/food_meg_analyses"

subs_directory = f"{project_directory}/SUBS_DIR"

subject_directory_pattern = f"{subs_directory}/sub*"

mat_file_path_pattern = f"*.mat"

html_report_path = f"report.html"

h5_report_path = f"report.h5"

def get_csd_path(condition):
    csd_path = f"csd_{condition}.h5"
    return csd_path

def get_csd_mean_path(condition):
    csd_mean_path = f"csd_mean_{condition}.h5"
    return csd_mean_path 

def get_report_titles(condition=None, contrast=None, fmin=None, fmax=None, tmin=None, tmax=None):
    report_titles = {'csd': f"CSD matrices for {condition}", 'csd_mean': f"CSD mean matrices for {condition}", 'coherence': f"Coherence mean matrices for {condition}", 
                    'tfr_contrast': f'Time-frequency representation for{contrast}', 'general_topoplots':f"across time topo-plots averaged for all conditions", 'gfp': f"Global Field Power",
                    'psd': 'Power Spectral Density for Evoked', 'tfr_contrast_topoplots':f"TFR Contrast Topoplot({contrast}, {fmin}-{fmax}Hz, {tmin}-{tmax}s)"}
    return report_titles

def get_report_sections(subject_num):
    report_sections = {'csd': f"{subject_num}/CSD", 'coherence': f"{subject_num}/Coherence", 'tfr_contrast':{subject_num}/'TFR contrast', 
                    'tfr_contrast_topoplots':f"{subject_num}/TFR Contrast Topoplots", 'gfp': f"{subject_num}/GFP", 'psd': f"{subject_num}/PSD"}
    return report_sections

epochs_path = "orig_epo.fif"

epochs_combined_path = "combined_epo.fif"

evoked_path = "evo.fif"

def get_tfr_contrast_path(con1, con2):
    evoked_tfr_contrast_path = f"evoked_tfr_{con1[0]}-{con2[0]}.h5"
    return evoked_tfr_contrast_path

psd_path = "psd.h5"

sub_dict_schema = {
        'data': {'type': 'dict', 
                
                'schema':

                    {
                    'trial':{'type': 'list', 
                             'schema': {'type': 'float'}},
                    'trialinfo':{'type': 'list',
                                 'schema': {'type': 'float'}},
                    'label':{'type': 'list', 'schema': {'type':'string'}},
                    'fsample':{'type': 'float'}
                    }
                }
        }


In [ ]:
def add_to_report(report: mne.Report, subject_num: str):
    epochs_combined = mne.read_epochs(glob.glob("*combined_epo.fif"))
    evoked  = mne.read_evokeds(glob.glob("*evo.fif"))
    
    #plot psd (computed for evoked): 
    psd = read_spectrum(psd_path)
    report.add_figure(psd.plot(), title=report_titles['psd'], section=report_sections['psd'], replace=True)
    
    #plot csds (computed for epochs_combined[condition]):
    for condition in [list(epochs_combined.event_id.keys()), 'baseline']:    
        csd = read_csd(csd_path)
        csd_mean = read_csd(csd_mean_path)
        report.add_figure(csd.plot(show=False), title= report_titles['csd'], section=report_sections.csd, replace=True)
        report.add_figure(csd_mean.plot(show=False), title= report_titles['csd_mean'], section=report_sections['csd'], replace=True)
        report.add_figure(csd_mean.plot(mode='coh', show=False), title= report_titles['coherence'], section=report_sections['coherence'], replace=True)

    #plot gfp for evoked:
    report.add_figure(evoked.plot_image(titles=f"Global Field Power for a single subject", show=False),title=report_titles['gfp'], 
    section=report_sections['gfp'], replace=True)

       #plot tfr contrast computed per contrast (evoked[condition_1] - evoked[condition_2]):
    for file in glob.glob("*tfr*.h5"):
            
        tfr_contrast = read_tfrs(file)
        contrast = file.split(['evoked_tfr_','.h5'])[1]
        report.add_figure(tfr_contrast.plot(combine='mean', baseline=baseline_time, title=f"Contrast ({contrast})"),
        title= report_titles['tfr_contrast'], section=report_sections['tfr_contrast'], replace=True)
        freqs = tfr_contrast.freqs
        freq_bands = np.arrange(freqs[0], freqs[-1]+1, 4) # np.ndarray

        #plot topo-plots of tfr specific contrast per time range and frequency band:
        for time_range in time_frames:
            for i,freq in enumerate(freq_bands):
                        
                # if reached to last frequency in numpy array -> no more fmin-fmax pairs to go over 
                if i == len(freq_bands)-1:
                    break
                fmin, fmax = freq, freq_bands[i+1]
                tmin, tmax = time_range[0], time_range[1]
            report.add_figure(tfr_contrast.plot_topomap(size=8, mode='mean', fmin=fmin, fmax=fmax, tmin=tmin, tmax=tmax, baseline=baseline_time), 
            title=report_titles['tfr_contrast_topoplots'], section=report_sections['tfr_contrast_topoplots'], replace=True)

In [ ]:
import mne

def combine_epochs(epochs: mne.EpochsArray, old_event_ids: dict, new_event_ids: dict)-> mne.EpochsArray|None:
    """
    Recieves:
    * epochs: mne epochs array instance
    * old_event_ids: dictionary in length of the number of conditions (18 in our case), must be divisible by 3.
      contains condition names as keys and the integer code for each conditio as values.
    * new_event_ids: dictionary in the length of the new number of conditions. must be in length - no. old conditions / 3.
      contains condition names as keys and the integer code for each conditio as values.

    Function:
    * Combines all conditions of same semantic category (food, positive, neutral) and repetition (presentation 1 and 2), 
    disregarding the lag (short, medium, long)

    Returns:
    * epochs_combined: mne.EpochsArray instance, with events categorized to the new conditions.

    Notes: 
    * A specific order of the conditions in old_event_ids and new_even_ids is required.
    """

    import mne
    import traceback
    import numpy as np
    from src import config
    from tests import input_validation_tests

    epochs_combined = None

    try:

        input_validation_tests.validation_func["combine_epochs"](epochs, old_event_ids, new_event_ids)
        

    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
        
    else:
        try:
            old_event_ids = list(old_event_ids.keys())
            num_keys_combined = len(old_event_ids)/len(new_event_ids)

            # goes through new event_ids and assignes a new event id for every triplet of old event ids and returns a new epochs array with combined
            # event ids
            for i in np.arange(len(new_event_ids)):
                epochs_combined = mne.epochs.combine_event_ids(epochs, 
                old_event_ids[num_keys_combined*i:num_keys_combined*i+num_keys_combined] , 
                {list(new_event_ids.keys())[i]: list(new_event_ids.values())[i]}, copy=True)

            epochs_combined.save(config.epochs_combined_path, overwrite = True)
        
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

    return epochs_combined


In [ ]:
def convert_mat_to_dict(file_name: str|os.PathLike) -> dict|None:
    """
    
    Recieves:
    * file_name: path to mat file for conversion. 
    
    Function:
    * Converts mat files from v. 7.3 to dictionaries and deals with possible exceptions.
    Returns: 
    * dict_from_mat: dictionary (if conversion was not successful, an exception occured, dict_from_mat = {})
    Notes:
    * For the following code to work the mat file should include epoched data.
        
    """
    import traceback
    from pymatreader import read_mat
    try:
        #validate input type:
        if not isinstance(file_name, (str, os.PathLike)):
            raise TypeError("file_name should be a directory to a mat file in str or PathLike format, input from another type was given")
    except Exception as e:
         print("An error occured:", e)
         traceback.print_exc()
    
    else:
            
        try:
            if os.path.exists(file_name):
                # using read_mat from pymatreader module, convert a v. 7.3 mat file to a dictionary
                dict_from_mat = read_mat(file_name)
            else:
                raise FileNotFoundError()
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()
            
    #returns dict_mat, if dict_mat doesn't exist (fail in conversion), return None
    if 'dict_from_mat' not in locals(): 
        dict_from_mat = None
    return dict_from_mat

In [ ]:
def extract_from_dict(sub_dict: dict) -> tuple[np.ndarray|None, np.ndarray|None, list[str]|None, float|None]:
    """
    
    Recieves:
    * sub_dict: subject dictionary with the data that was converted from mat file, with fields:['data']['trial'], ['data']['trialinfo'], 
    ['data']['grad']['label'], ['data']['fsample'].
    
    Function:
    * Extracts relevant data from the dictionary for the following steps.
    Returns: 
    * data: ndarray of shape (trials, channels, time points). 
    * events_code: ndarray of shape (1, trials), stores the integers corresponding to the condition tested in each trial, 
    * ch_names: list of length 246, first 246 channel (sensor) names as a  (all "good" magnometers, the bad and reference channels are excluded).
    * s_freq: int, sampling frequency.
     """
    
    from tests import input_validation_tests
    import traceback
    try:
        
        input_validation_tests.validation_func['extract_from_dict'](sub_dict)
    except ValueError as e:
        print("An error occured:")
        print("sub_dict has missing keys or incorrect types of values, check dictionary or change extract_from_dict function")
        
    except Exception as e:
        print("An error occured:", e)
    else:
        try:
            # Extract data and trial info
            data = sub_dict.get("data").get("trial") 
            events_code = np.array(sub_dict.get("data").get("trialinfo")[:, 0], dtype=int) # convert from float to int
            ch_names = sub_dict.get("data").get("label")
            sfreq = sub_dict.get("data").get("fsample")
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

    #return data, events_code, ch_names, sfreq , if one of the variables doesn't exist (an exception occured during the process), return an empty variable / 0.
    if 'data' not in locals():
        data = None
    if 'events_code' not in locals():
        events_code = None
    if 'ch_names' not in locals():
        ch_names = None
    if 'sfreq' not in locals():
        sfreq = None
    return data, events_code, ch_names, sfreq

In [ ]:
def remove_oddball_trials(data: np.ndarray, events_code: np.ndarray, oddball_id: int) -> tuple[np.ndarray|None, np.ndarray|None]:
    """ 
    
    Recieves:
    * data: numpy ndarray, shape (trials, channels, time points).
    * events_code: numpy ndarray of type int, shape (1, trials), contains unique code for each stimuls.
    * odball_id: integer, code of the odball stimulus.
    Function:
    * Removes all trials coressponding to oddball stimulus id.
    Reutrns:
    * data: numpy ndarray shape (trials, channels, time points), after oddball trial removal.
    * events_code: numpy ndarray of type int, shape (1, trials), after oddball id removal.
    
    """
    from tests import input_validation_tests
    import traceback
    try:
        input_validation_tests.validation_func['remove_oddball_trials'](data, events_code, oddball_id)
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:
            
            oddball_idx = np.where(events_code == oddball_id)[0]  # using [0] to extract the indices array of the oddball stimulus,
            # those are the indices of the trials in data corresponding to the oddball stimulus

            # Remove odball code from events_code + oddball trials from data
            events_code_removed = np.delete(events_code, oddball_idx)
            data_removed = np.delete(data, oddball_idx, axis=0)  # axis=0 --> remove odball from data rows (trials)

        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()
            
    #return data_removed, if data_removed doesn't exist (an exception occured during the process), return an empty ndarray
    if 'data_removed' not in locals():
        data_removed = None
    if 'events_code_removed' not in locals():
        events_code_removed = None  
    return data_removed, events_code_removed

In [ ]:
def create_events_for_epochs(events_code: np.ndarray) -> np.ndarray|None:
    """
    Recieves:
    * events_code: numpy ndarray of integers with the code for each condition
    Function:
    * Creates an events (trials, 3) numpy ndarray.
        First column: The first column contains the event onset, because data already epoched - 
        np.arange(len(events_code), dtype=int), onset time should be different otherwise events is not valid input to EpochsArray.
        Second column: The second column contains the signal value of the immediately preceding sample, 
        and reflects the fact that event arrays sometimes originate from analog voltage channels.
        In most cases it is all zeros and can be ignored.
        Third column: events code for each trial according to condition.
    Reutrns:
    * events: 3D array with events ready to be input in mne.EpochsArray
    """
    from tests import input_validation_tests
    import traceback
    try:
        input_validation_tests.validation_func['create_events_for_epochs'](events_code)
        
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:
                # Create event onset and preceding event arrays for the 3D events structure required in MNE epochs class:
                event_onset = np.arange(len(events_code), dtype=int)
                event_precede = np.zeros(len(events_code), dtype=int)
                
                # Stack the event info into the correct shape and structure for an epochs array input (onset, preceding, events_code)
                events = np.vstack((event_onset, event_precede, events_code)).T
        
        except Exception as e:
                print("An error occured:", e)
                traceback.print_exc()
    if 'events' not in locals():
            events = None
    return events    

In [ ]:
def convert_dict_to_epochs(sub_dict: dict, mne_info: mne.Info) -> tuple[mne.EpochsArray|None, mne.EvokedArray|None]:
    """
    Recieves:
    * sub_dict: subject dictionary with the already epoched data that was converted from mat file, with fields:['data']['trial'], ['data']['trialinfo'], 
    ['data']['grad']['label'], ['data']['fsample'].  
    * mne_info: instance of mne.Info class
    Function:
    * Converts dictionary to MNE epochs array.
    Reutrns:
    * MNE epochs array, or an empty dictionary in case of an exception
    """
    from src import config
    from tests import input_validation_tests
    import traceback
    try:
        input_validation_tests.validation_func['convert_dict_to_epochs'](sub_dict, mne_info)
        
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:      
            # variables imported from config.py:
            tmin = config.baseline_time[0] # starting point of baseline (-0.3 in our case) 
            baseline = config.baseline_time # tuple for baseline time (-0.3,0)
            oddball_id = oddball_id # int of code for oddball stimulus
            data, events_code = extract_from_dict(sub_dict)

            # Identify and remove oddball trials
            data, events_code =  remove_oddball_trials(data, events_code, oddball_id)
            events = create_events_for_epochs(events_code)

            # Create the epochs instance:
            epochs = mne.EpochsArray(data, mne_info, events=events, tmin=tmin, event_id=config.event_ids,
                reject=None, flat=None, reject_tmin=None, reject_tmax=None,
                baseline=baseline, proj=True, on_missing='raise', metadata=None,
                selection=None, drop_log=None, raw_sfreq=None, verbose=None)
            evoked = epochs.average()
                            
            # save the epochs arrays in the current subject's folder:
            epochs.save(config.epochs_path, overwrite=True)
            evoked.save(config.evoked_path, overwrite=True)
        except Exception as e:
            print(" An error occured:", e)
            traceback.print_exc()

    #returns epochs, if epochs doesn't exist due to exception, return an empty dictionary
    if 'epochs' not in locals():
        epochs = None
    if 'evoked' not in locals():
        evoked = None
    return epochs, evoked  

In [ ]:
def convert_mat_to_epochs(file_name: os.PathLike, info = None) -> tuple[mne.EpochsArray|None, mne.evoked.Evoked|None]:
    """
    Recieves:
    * file_name: mat file path to convert to an mne.EpochsArray instance
    * info: mne.Info instance, if info is not given a manual info is created (the manuall info doesn't contain sensor positions)
    Function:
    * Convert mat structure to an EpochsArray instance.
    Reutrns:
    * epochs: EpochsArray instance, or an empty dictionary in case of exception
    """
    from src import config
    from mat_to_epochs_conversion import create_info
    import traceback
    from tests import input_validation_tests
    
    try: 
        input_validation_tests.validation_func['convert_mat_to_epochs'](file_name, info)
        
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:
            if not os.path.exists(file_name):
                raise FileNotFoundError(f"The file: {file_name}, doesn't exist.")
            
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()
        else: 
            try:
                sub_dict = convert_mat_to_dict(file_name)
                if info is None:
                    mne_info = create_info.create_mne_info(sub_dict)
                    
                epochs, evoked = convert_dict_to_epochs(sub_dict, mne_info)
            except Exception as e:
                print("An error occured:", e)
                traceback.print_exc()
            
    
    #returns epochs and evoked, if doesn't exist due to exception, return None
    if 'epochs' not in locals():
        epochs = None
    if 'evoked' not in locals():
        evoked = None
    return epochs, evoked

In [ ]:
import os, mne
# in case a raw object exists:
def extract_raw_info(folder_directory: os.PathLike) -> mne.Info|None:
    """
    Recieves:
    * folder_directory: directory to the folder where the raw MEG bti recording is saved
    Function:
    * reads and extracts info from raw MEG bti recording
    Returns:
    * mne.Info instance
    """
    import traceback, glob
    from src import config
    try:
        if not isinstance(folder_directory, (str, os.PathLike)):
            raise TypeError("folder_directory should be a directory in a str or PathLike format containing raw MEG bti file, \n \
                 input from another type was given")
    except TypeError as e:
        print("Type Error:", e)
        traceback.print_exc()
    else:
        try:
            if os.path.exists(folder_directory):
                # redirect to the folder
                os.chdir(folder_directory) 
                # glob.glob returns a list of the paths with the desired pattern, return the first and only object in the list
                raw_path = glob.glob(f"*1Hz")[0] 
                print(raw_path)
                # read raw object
                raw = mne.io.read_raw_bti(raw_path, rename_channels=False)
                # drop all bad channels and reference channels (leaves 246 channels)
                raw.drop_channels(config.bad_ch_names)
                
                # extracts only info from raw object
                raw_info = raw.info
            
            else: 
                raise FileNotFoundError(f"Directory {folder_directory} doesn't exist.")
            
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()
    
    #returns raw_info, if doesn't exist due to exception, return an empty dictionary
    if 'raw_info' not in locals():
        raw_info = None
    return raw_info

In [ ]:
def create_mne_info(sub_dict: dict) -> mne.Info|None:
    """
    Recieves:
    * sub_dict: dictionary with fields ['data']['trial'], ['data']['trialinfo'], 
    ['data']['grad']['label'], ['data']['fsample'].  
    Function:
    * Creates a manual mne.Info instance with info: channel names, channel_types, sampling frequency
    Returns:
    * mne_info: an instance of mne.Info object, or an empty dictionary in case of an exception.
    """
    import numpy as np
    from src import config
    from mat_to_epochs_conversion.convert_main_funcs import extract_from_dict
    import traceback
    try:
        #validate input type:
        if not isinstance(sub_dict, dict):
            raise TypeError("sub_dict should be a dictionary, another input type was recieved")
    
    except Exception as e:
        print("An error has occured:", e)
    else:
        try:
            _, _, ch_names, sfreq = extract_from_dict(sub_dict)
            ch_types = np.array(config.channels_number * ['mag']) # create 246 'mag' channel types relating to the 246 extracted channel names in extract_from_dict channels 
            mne_info = mne.create_info(ch_names, sfreq, ch_types, verbose=None)
        
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

    #returns mne_info, if doesn't exist due to exception, return None
    if mne_info not in locals():
        mne_info = None
    return mne_info

In [ ]:
def compute_csd(epochs_instance: mne.EpochsArray, condition:str, freq_bands: list, time_range: tuple) \
    -> tuple[mne.time_frequency.CrossSpectralDensity|None, mne.time_frequency.CrossSpectralDensity|None]:
    """
    Recieves:
    * epochs_instance: mne.EpochsArray.
    * condition: str, the event_id key present in epochs_instance that corresponds to the experimental condition.
    * freq_bands: list of tuples(1,2) containing the lower an upper bound for each frequency band.
    * time_range: tuple, post stimulus / baseline time range.
    Function:
    * Calculate the cross spectral density for all channels in epochs through the set frequencies for the whole time range.
      for a specific epochs condition using morlet wavelet.  
    Returns:
    * csd: CrossSpectralDensity instance, the cross spectral density calculated.
    """
    import traceback
    from src import  config
    from mne.time_frequency import csd_morlet
    from tests import input_validation_tests
    try:
        input_validation_tests.validation_func['compute_csd'](epochs_instance, condition, freq_bands, time_range)
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:
        # set the time and frequency range for csd calculation (whole time and frequency range):
            tmin = min(time_range)
            tmax = max(time_range)
            fmin = freq_bands[0][0]
            fmax = freq_bands[-1][1]
            frequencies = np.arange(fmin, fmax + 1, 2) # calculate the csd for the frequencies in the frequency range with a 2Hz step

            # epochs_baselined = epochs_instance[condition].apply_baseline((csd_tmin, csd_tmax)) # see what yields without and with baseline

            # extracts the epochs data for a single condition, the condition in which we desire to compute the csd.
            epochs_for_csd = epochs_instance[condition]
            
            # Compute CSD for the desired time interval and frequencies
            csd = csd_morlet(epochs_for_csd, frequencies=frequencies, tmin=tmin,
                            tmax=tmax, decim=20, n_jobs=-1, verbose=True)
            
            # average csds over frequency bands, each frequency band is a tuple (f[0], f[1])
            csd_mean = csd.mean([f[0] for f in freq_bands], [f[1] for f in freq_bands])

            # save original and mean csd:
            csd.save(config.get_csd_path(condition)) 
            csd_mean.save(config.get_csd_mean_path(condition))
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

    if 'csd' not in locals():
        csd = None
    if 'csd_mean' not in locals():
        csd_mean = None       
    return csd, csd_mean

In [ ]:
def compute_tfr_contrast(epochs: mne.EpochsArray, freqs: np.ndarray, con1: tuple, con2: tuple)\
    -> mne.time_frequency.AverageTFR|None:
    """
    Recieves:
    * epochs: mne.EpochsArray object
    * freqs: 1D-array, a range of numbers defining the start, end, and step frequencies.
    * con1: tuple, tuple[0] - name of first combined condition to contrast, 
      tuple[1] - a list of str of the name of conditions present in epochs combined under the same new condition -> tuple[0]
    * con2: tuple, tuple[0] - name of second combined condition to contrast, 
      tuple[1] - a list of str of the name of conditions present in epochs combined under the same new condition -> tuple[0]
    * report: mne.Report instance
    Funtion:
    * Add plots of Time-Frequency Representation (TFR) of the contrast (con1-con2) between two conditions.
    Returns: 
    * Time-Frequency Representation (TFR) for the epochs (con1-con2) contrast.
    """
    import traceback
    from src import config
    from tests import input_validation_tests

    # input handling
    try:
        input_validation_tests.validation_func['compute_tfr_contrast'](epochs, freqs, con1, con2)
    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
    else:
        try:
            # Duplicate the epochs and work with the copy
            epochs_copy = epochs.copy()

            # Take the average within each condition
            epochs_con_1 = epochs_copy[con1[1]].average()
            epochs_con_2 = epochs_copy[con2[1]].average()

            # Subtract the data (assuming the data shapes are the same)
            contrast = epochs_con_1.data - epochs_con_2.data

            # Create a new info object, assuming the same channels and info as the original EvokedArrays
            info = epochs_con_1.info 

            # Create a new Evoked object with the contrast data
            evo_contrast = mne.EvokedArray(contrast, info, tmin=epochs_con_1.tmin)

            # Compute TFR
            try:
                tfr_contrast = evo_contrast.compute_tfr(method='morlet', tmin=config.baseline_time[0], tmax=config.post_stim_time[1], freqs=freqs)
                tfr_contrast.save(config.get_tfr_contrast_path(con1, con2))
            except ValueError as e:
                print(f"Error in computing the TFR: {e}\nAdjust the frequency range. freqs=(8, 24, 2) works best!")
            except Exception as e:
                print("An error occured:", e)
                traceback.print_exc()
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()
        
    if 'tfr_contrast' not in locals():
        tfr_contrast = None
    return tfr_contrast

In [ ]:
def psd(evoked_instance: mne.evoked.Evoked)-> mne.time_frequency.Spectrum:
    from src import config
    import traceback
    
    try:
        if not isinstance(evoked_instance, mne.evoked.Evoked):
            raise TypeError("evoked_instance should be an mne.evoked.Evoked instance, wrong input type was given.")
        
    except TypeError as e:
        print("An error occured:", e)
    else:
        try:
            psd = evoked_instance.compute_psd(method='morlet', fmin=2, fmax=30, tmin=config.baseline_time[0], tmax=config.post_stim_time[1], picks=['meg'])
            psd.save(config.psd_path)
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

In [ ]:
import mne

def combine_epochs(epochs: mne.EpochsArray, old_event_ids: dict, new_event_ids: dict)-> mne.EpochsArray|None:
    """
    Recieves:
    * epochs: mne epochs array instance
    * old_event_ids: dictionary in length of the number of conditions (18 in our case), must be divisible by 3.
      contains condition names as keys and the integer code for each conditio as values.
    * new_event_ids: dictionary in the length of the new number of conditions. must be in length - no. old conditions / 3.
      contains condition names as keys and the integer code for each conditio as values.

    Function:
    * Combines all conditions of same semantic category (food, positive, neutral) and repetition (presentation 1 and 2), 
    disregarding the lag (short, medium, long)

    Returns:
    * epochs_combined: mne.EpochsArray instance, with events categorized to the new conditions.

    Notes: 
    * A specific order of the conditions in old_event_ids and new_even_ids is required.
    """

    import mne
    import traceback
    import numpy as np
    from src import config
    from tests import input_validation_tests

    epochs_combined = None

    try:

        input_validation_tests.validation_func["combine_epochs"](epochs, old_event_ids, new_event_ids)
        

    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()
        
    else:
        try:
            old_event_ids = list(old_event_ids.keys())
            num_keys_combined = len(old_event_ids)/len(new_event_ids)

            # goes through new event_ids and assignes a new event id for every triplet of old event ids and returns a new epochs array with combined
            # event ids
            for i in np.arange(len(new_event_ids)):
                epochs_combined = mne.epochs.combine_event_ids(epochs, 
                old_event_ids[num_keys_combined*i:num_keys_combined*i+num_keys_combined] , 
                {list(new_event_ids.keys())[i]: list(new_event_ids.values())[i]}, copy=True)

            epochs_combined.save(config.epochs_combined_path, overwrite = True)
        
        except Exception as e:
            print("An error occured:", e)
            traceback.print_exc()

    return epochs_combined

In [5]:
"""

Itterate over all subjects folders, convert mat files of epoched data to EpochsArray instances, combine epochs
to the desired conditions and save the two epochs arrays to fif files.
Analyse epochs_combined data using CSD, TFR, GFP, PSD and topo plots. 

Subjects foldes must contain only one mat file that contains the epoched data and one raw MEG bti recording.

"""
"""importations of libraries"""

if __name__ == "__main__":

    if package_path not in sys.path:
        sys.path.insert(0, package_path)
   

    # in case one of the modules is not installed or can not be found by python using the system variables:
    # except Exception as e:
    #     print("An error occured:", e)
    #     traceback.print_exc()

    try:

        directory = config.subject_directory_pattern # directory pattern for itterating over subject folders

        # itterate over all subjects folders
        for folder in glob.iglob(directory): 

            if os.path.exists(folder): # the subject folder that contains the mat file with epoched data and the raw MEG recordings per subject

                try:  

                    subject_num = folder.split("SUBS_DIR\\", 1)[1]

                    os.chdir(folder)

                    report = mne.Report(title=f"report for {subject_num}")

                    raw_info = create_info.extract_raw_info(folder)
                    
                    # recieves a file path to the mat file, glob.glob returns a list of all paths found with the pattern.
                    # [0] for returning the first and only element in the list.
                    epochs, evoked = convert_main_funcs.convert_mat_to_epochs(glob.glob(config.mat_file_path_pattern)[0], info=raw_info) 
                    
                    # combine epochs by new conditions (new_event_ids):
                    epochs_combined = combine_epochs.combine_epochs(epochs, config.event_ids, config.new_event_ids)
                
                    # extract conditions for csd cmputation per condition:
                    conditions =  list(epochs_combined.event_id.keys())
                
                    # Suppress warning about wavelet length.
                    warnings.simplefilter('ignore')

                    for condition in conditions:

                        # csd calculation post stimulus over the desired frequency range per condition, save and add to report
                        csd = compute_csd.compute_csd(epochs_combined, condition, config.freq_bands, config.post_stim_time) 
                        
                    # csd calculation of baseline over the desired frequency range, save and add to report. (calculates csd baseline for the last 
                    # condition in loop, we assume that all conditions have same baseline activity)
                    csd_baseline = compute_csd(epochs, condition, config.freq_bands, config.baseline_time)

                    
                    # compute tfrs for desired contrast of conditions, over the frequencies in freqs and save:
                    tfr = tfr_psd_analyses.compute_tfr_contrast(epochs=epochs, subject_num=subject_num, freqs=np.arange(8, 24, 2), con1=('pres_1', config.pres_1), 
                    con2=('pres_2', config.pres_2), report=report)


                    tfr = tfr_psd_analyses.compute_tfr_contrast(epochs=epochs, subject_num=subject_num, freqs=np.arange(8, 24, 2), con1=('food',config.food), 
                    con2=('nonfood',config.nonfood), report=report)

                        
                except Exception as e:
                    print("An error occured:", e)
                    traceback.print_exc()


            else:
                raise FileNotFoundError

    except Exception as e:
        print("An error occured:", e)
        traceback.print_exc()


ModuleNotFoundError: No module named 'src'